In [178]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


In [179]:
img = cv2.imread('data/image0.png')
mask = cv2.imread('data/mask0.png', 0)

#invertion masque
mask = cv2.bitwise_not(mask)

mask = cv2.bitwise_not(mask)

# 2. On crée un "stack" (un paquet) des deux images
# Pour que ça marche, le masque doit avoir 3 canaux comme l'image
mask_3ch = cv2.merge([mask, mask, mask])
combined = np.stack([cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask_3ch])

# 3. Affichage avec Plotly Express
# facet_col=0 dit à Plotly de séparer le "stack" en deux colonnes
fig = px.imshow(combined, facet_col=0, binary_string=True)

# 4. On nomme les axes et les titres
fig.layout.annotations[0].text = "Image Originale"
fig.layout.annotations[1].text = "Masque"

fig.update_layout(showlegend=False)
fig.show()



In [180]:
# pix_mask =[]
# for i in range(len(mask)) :
#     for j in range(len(mask[0])):
#         if mask[i,j] < 100 :
#             pix_mask.append([i,j])
# print(pix_mask)

mask = cv2.resize(mask, (img.shape[1], img.shape[0]))  # d'abord

pix_mask = []
for i in range(mask.shape[0]):
    for j in range(mask.shape[1]):
        if mask[i,j] < 100:
            pix_mask.append([i,j])

mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
# mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)


# 3. Récupération du masque binaire (on prend l'index [1])
_, mask_only = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

# 4. Application du masque
# On fait un "ET" entre l'image et elle-même, mais UNIQUEMENT là où le masque est blanc
img_m = cv2.bitwise_and(img, img, mask=mask_only)

# 5. Affichage Plotly Express
# Conversion BGR -> RGB pour que les couleurs soient les bonnes
img_m_rgb = cv2.cvtColor(img_m, cv2.COLOR_BGR2RGB)
fig = px.imshow(img_m_rgb)
fig.update_layout(title="Aperçu de la zone masquée (Coordonnées actives)")
fig.show()




In [181]:
#algo

#noyeau 3 3
k = np.array([[1, 1, 1],
              [1, 0, 1],
              [1, 1, 1]])/8


k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4],  
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
])  
k_5x5 = k_5x5 / k_5x5.sum()  # normalisation propre



# def algo_oliveira(img, mask_indices):
#     image = img.astype(np.float32)
#     # On crée un masque binaire (1 pour image saine, 0 pour trou)
#     mask_map = np.ones(image.shape[:2], dtype=np.float32)
#     for y, x in mask_indices:
#         mask_map[y, x] = 0

#     for i in mask_indices:
#         y, x = i[0], i[1]
        
#         if 2 <= y < image.shape[0] - 2 and 2 <= x < image.shape[1] - 2:
#             # 1. On extrait les voisins de l'image ET du masque binaire
#             voisins = image[y-2:y+3, x-2:x+3]
#             voisins_valides = mask_map[y-2:y+3, x-2:x+3] # 1 si sain, 0 si masque
            
#             # 2. On ajuste le noyau : on ne garde que les poids des pixels SAINS
#             # On multiplie le noyau par le masque de validité
#             noyau_adapte = k_5x5 * voisins_valides
            
#             # 3. Normalisation CRUCIALE : 
#             # La somme des poids doit toujours être 1, mais seulement sur les pixels valides
#             somme_poids = np.sum(noyau_adapte)
            
#             if somme_poids > 0:
#                 noyau_normalise = noyau_adapte / somme_poids
#                 for c in range(3):
#                     image[y, x, c] = np.sum(voisins[:, :, c] * noyau_normalise)
    
#     return image.astype(np.uint8)






mask_bool = np.zeros(img_m_rgb.shape[:2], dtype=bool)
for y, x in pix_mask:
    mask_bool[y, x] = True

k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4], 
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
], dtype=np.float32) / 100

In [182]:
def priotité(patch, patch_size):
    """
    Calcule le score de priorité pour UN SEUL patch donné.
    Le patch passé en entrée est l'image masquée (où le trou vaut 0).
    """
    # 1. TERME DE CONFIANCE C(p)
    # Les pixels sains sont ceux qui ne sont pas égaux à 0
    pixels_sains = (patch != 0)
    
    # La confiance est la proportion de pixels sains dans le patch
    nb_pixels_sains = np.sum(pixels_sains)
    surface_totale = patch_size * patch_size
    confiance = nb_pixels_sains / surface_totale
    
    # 2. TERME DE DONNÉE D(p)
    # On calcule les gradients (contours) horizontaux et verticaux du patch
    # On utilise Sobel pour l'intensité des contours
    grad_x = cv2.Sobel(patch, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(patch, cv2.CV_64F, 0, 1, ksize=3)
    
    # On prend la magnitude maximale du gradient (la force du contour dans le patch)
    magnitude_gradients = np.sqrt(grad_x**2 + grad_y**2)
    donnee = np.max(magnitude_gradients)
    
    # 3. SCORE FINAL : Priorité = Confiance * Donnée
    priorite = confiance * donnee
    
    return priorite

In [ ]:


mask = cv2.bitwise_not(mask)

def algo_criminisi(image, mask, largeur_patch):
    mask_bool = np.zeros(img_m_rgb.shape[:2], dtype=bool)
    for y, x in pix_mask:
        mask_bool[y, x] = True
    

    gros_noyau = np.ones((2 * largeur_patch - 1, 2 * largeur_patch - 1), np.uint8)


    
    
    #dimension image
    h, w = image.shape[:2] 
    marge = largeur_patch -1
    iter_count = 0
    while mask_bool.any(): 
        #compteur iterations
       
        iter_count += 1
        print(f"Iteration {iter_count} - Pixels restants : {np.sum(mask_bool)}")
        
        
        zone_interdite = cv2.dilate(mask, gros_noyau, iterations=1)
        
        
        
        
        frontiere, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        points_frontiere = np.vstack(frontiere)
        liste_pixels_frontiere = points_frontiere[:, 0]
       
       # Vérifier ce qu'on récupère
        print(f"Nombre de contours : {len(frontiere)}")
        for i, c in enumerate(frontiere):
            print(f"  Contour {i} : {len(c)} points, premier point : {c[0]}")
       
       
       #init
        pix_coord = [0,0]
        prio = -1
        R = largeur_patch - 1  
        #priotité
        for x, y in liste_pixels_frontiere:
                    
            ymin = y - R
            ymax = y + R + 1
            xmin = x - R
            xmax = x + R + 1
         
         
            if not (ymin < 0 or ymax > h or xmin < 0 or xmax > w):
               
                patch = image[ymin:ymax, xmin:xmax]
                new_prio = priotité(patch, largeur_patch)
                if new_prio > prio:
                    prio = new_prio
                    pix_coord = [x, y]
            
        # #recherche du patch de correction adapté
        # for x,y in image:
            
        #     ymin = y - R
        #     ymax = y + R + 1
        #     xmin =  x - R
        #     xmax = x + R + 1
        #     if  not(ymin < 0 or ymax > h or xmin < 0 or xmax > w):
                
        #          # Slicing
        #         patch_correction = image[ymin:ymax, xmin:xmax]
            
         
       # Extraction patch cible
        x, y = pix_coord[0], pix_coord[1]
        ymin, ymax = y - R, y + R + 1
        xmin, xmax = x - R, x + R + 1

        patch_cible        = image[ymin:ymax, xmin:xmax]
        patch_mask_region  = mask[ymin:ymax, xmin:xmax]
        indices_patch_mask = (patch_mask_region > 0)   # pixels à remplir
        pixels_connus      = ~indices_patch_mask        # pixels utilisables

        meilleur_score = float('inf')
        meilleur_patch = None

        for py in range(marge, h - marge):
            for px in range(marge, w - marge):

                if zone_interdite[py, px]:
                    continue

                patch_corr = image[py - R : py + R + 1, px - R : px + R + 1]

                # SSD uniquement sur pixels connus
                score = np.sum((patch_cible[pixels_connus] - patch_corr[pixels_connus]) ** 2)

                if score < meilleur_score:
                    meilleur_score = score
                    meilleur_patch = patch_corr.copy()

        
        #maj
        ys, xs = np.where(mask[ymin:ymax, xmin:xmax] > 0)
        ys_abs, xs_abs = ys + ymin, xs + xmin
        image[ys_abs, xs_abs]     = meilleur_patch[ys, xs]
        mask_bool[ys_abs, xs_abs] = False
        mask[ys_abs, xs_abs]      = 0

In [184]:
resultat = algo_criminisi(img_m_rgb, mask, largeur_patch=9)
px.imshow(img_m_rgb).update_layout(title="Image depart").show()
px.imshow(resultat).update_layout(title="Image réparée").show()

Iteration 1 - Pixels restants : 2713
Nombre de contours : 2
  Contour 0 : 280 points, premier point : [[136 384]]
  Contour 1 : 129 points, premier point : [[119 346]]
Iteration 2 - Pixels restants : 2639
Nombre de contours : 2
  Contour 0 : 280 points, premier point : [[136 384]]
  Contour 1 : 126 points, premier point : [[119 346]]
Iteration 3 - Pixels restants : 2536
Nombre de contours : 2
  Contour 0 : 280 points, premier point : [[136 384]]
  Contour 1 : 132 points, premier point : [[119 346]]
Iteration 4 - Pixels restants : 2502
Nombre de contours : 2
  Contour 0 : 280 points, premier point : [[136 384]]
  Contour 1 : 116 points, premier point : [[126 346]]
Iteration 5 - Pixels restants : 2435
Nombre de contours : 2
  Contour 0 : 280 points, premier point : [[136 384]]
  Contour 1 : 109 points, premier point : [[126 346]]
Iteration 6 - Pixels restants : 2411
Nombre de contours : 2
  Contour 0 : 280 points, premier point : [[136 384]]
  Contour 1 : 90 points, premier point : [[141

TypeError: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''